<div align="center">

#### [S. Mussard](https://sites.google.com/view/cv-stphane-mussard/accueil "Homepage") 

</div>

<div align="center">

#### Chapter 5: Multiple Gini Regressions 

</div>


<div align="center"> <a href="https://www.python.org/"><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f8/Python_logo_and_wordmark.svg/390px-Python_logo_and_wordmark.svg.png" style="max-width: 150px; display: inline" alt="Python"/></a> 

</div>


<div align="center"> </div>

<div align="center">

Cite: S. Mussard (2025), *Machine Learning with Gini Indices: Applications with Python*, Springer.  

</div>



# Application Gini-DF Tests

In [ ]:
# Install arch for unit root tests
#!pip install arch 

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime as dt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
import matplotlib.pyplot as plt
import scipy.stats as ss


## Data: Airlines Passengers

Box G. E. P., Jenkins G. M., Reinsel G. C. (1994). Time Series Analysis: Forecasting and Control, Third edition. Prentice Hall. ISBN 978-0130607744. 

In [ ]:
# Airline passengers dataset from Box and Jenkins et al. 1976
base_air = pd.read_csv("AirPassengers.csv", sep=',')
base_air


In [ ]:
# Plot
plt.plot(base_air['#Passengers'])
plt.title("Airline passengers")


### Treat seasonality: decomposition of the series

In [ ]:
# Multiplicative scheme
decomposition = sm.tsa.seasonal_decompose(base_air['#Passengers'], model='multiplicative', period=12, extrapolate_trend='freq')
# Saesonality (coef)
season = decomposition.seasonal
trend = decomposition.trend
resid = decomposition.resid
# Plot
decomposition.plot()
plt.show()

In [ ]:
# ADF test of the trend
ADF_test = adfuller(trend, autolag='AIC', regression='ct')
result = pd.Series(ADF_test[0:4], index=['Test Statistic','p-value','Lags','Number of observations'])
for idx, value in ADF_test[4].items():
    result['Critical Value (%s)'%idx] = value
print (result)

In [ ]:
# ADF test of the residuals
ADF_test = adfuller(resid, autolag='AIC', regression='c')
result = pd.Series(ADF_test[0:4], index=['Test Statistic','p-value','Lags','Number of observations'])
for idx, value in ADF_test[4].items():
    result['Critical Value (%s)'%idx] = value
print (result)

In [ ]:
#!pip install arch
import arch
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron

# Phillips-Perron Test using arch
pp_test = PhillipsPerron(resid)
pp_result = pp_test.stat, pp_test.pvalue, pp_test.critical_values
print("PP Test Statistic:", pp_result[0])
print("PP p-value:", pp_result[1])
print("PP Critical Values:", pp_result[2])

# KPSS Test
kpss_result = kpss(resid, regression='c')  # 'c' for constant, 'ct' for trend
print("KPSS Test Statistic:", kpss_result[0])
print("KPSS p-value:", kpss_result[1])
print("KPSS Critical Values:", kpss_result[3])

## Testing autocorrelations

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(np.arange(0,11), acf(resid, nlags = 10), label='ACF')
plt.title('Autocorrelation function')
plt.xlabel('Lags')
plt.ylabel('ACF value')
plt.legend()

In [ ]:
# Ljung-Box Test
from statsmodels.stats.diagnostic import acorr_ljungbox
ljung_box_results = acorr_ljungbox(resid, return_df=True)
print(ljung_box_results)

# DF-Gini Test

In [ ]:
def autocorrel_gini(x, k):
    """
    Calculate the Gini autocorrelation for the series x for a given lag k.
    Return: Gini autocorelation coef
    """
    # Ranks of x
    r_x = ss.rankdata(x, method='average') / len(x)
    # Calculate Gini autocorel
    if k == 0:
        return 1
    else:
        n = np.sum((x[k:] - np.mean(x)) * (r_x[:-k] - np.mean(r_x)))
        d = np.sum((x - np.mean(x)) * (r_x - np.mean(r_x)))
        if n/d == 0:
            return np.nan 
        return n/d

In [ ]:
# Gini ACF
list_acf = []
for i in range(11):
    list_acf.append(autocorrel_gini(resid, i))
plt.figure(figsize=(8, 6))
plt.plot(np.arange(0,11), acf(resid, nlags = 10), label='ACF')
plt.plot(np.arange(0,11), list_acf, label='Gini ACF')
plt.title('Correlograms')
plt.xlabel('Lags')
plt.ylabel('ACF values')
plt.legend()

In [ ]:
# Gini ADF Test
from statsmodels.tsa.adfvalues import mackinnoncrit, mackinnonp
from GiniRegression_v2 import GiniRegression
def adf_gini(x):
    x_1 = x.shift(1) # x_{t-1}
    x_1 = x_1.dropna()
    y = np.array(x[1:])
    r_x = ss.rankdata(x_1, method='average') 
    model =  GiniRegression()
    model.fit(y, x_1)
    adf_test_gini = (model.beta_coeff[1] - 1) / model.stdev_coeff[1]
    p_value = mackinnonp(adf_test_gini, regression='c')
    return print("ADF Gini Stat:", adf_test_gini, "\n", "p_value:", p_value)
adf_gini(resid)
    